## Emergency Medicine Triage - Basic Prompt (without RAG)

This notebook implements a BASIC LLM Analysis for emergency medicine triage decision support. The system processes new patient cases by:
1. Analyzing the case description
2. Referecing Basic Triage Category Definitions
3. Generating a triage assessment


****NO RAG is implemented**


In [ ]:
import json
import re
from typing import List, Dict, Any, Optional, Tuple, TypedDict
import os
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from tqdm.notebook import tqdm
import traceback

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import JsonOutputParser

# LangGraph imports
from langgraph.graph import StateGraph, END

## Step 1: Initial Setup and Configuration

This section initializes all required components:
   
1. **LLM Setup**(choose one): Used to generate clinical summaries and reasoning task
    1. **DeepSeek-v3**
    2. **gpt-4o**
    3. **claude-3-7-sonnet**

2. **State Definition**:
   - Defining the `GraphState` class to track data through the pipeline
   - Managing transitions between processing stages

In [ ]:
# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "DEEPSEEK_API_KEY",
    "AZURE_OPENAI_LLM_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
    "ANTHROPIC_API_KEY",
    "ANTHROPIC_MODEL_NAME",
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing environment variables: {', '.join(missing_vars)}")

print("Environment variables loaded successfully.")

In [ ]:
def initialize_deepseek(
    api_key: str = os.getenv("DEEPSEEK_API_KEY"),
    base_url: str = "https://api.deepseek.com",
    model_name: str = "deepseek-chat",
    temperature: float = 0
):
    """
    Initialize DeepSeek client using LangChain
    
    Args:
        api_key (str): DeepSeek API key
        base_url (str): DeepSeek API base URL
        model_name (str): Model name to use
        temperature (float): Temperature for generation
        
    Returns:
        ChatOpenAI: Initialized DeepSeek client
    """
    return ChatOpenAI(
        model=model_name,
        api_key=api_key,
        base_url=base_url,
        temperature=temperature
    )

llm = initialize_deepseek()

print("LLM initialized successfully.")

model = "deepseekv3"

In [ ]:
from langchain_openai import AzureChatOpenAI


def initialize_azure_openai(
    azure_deployment_name: str = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME"),
    azure_endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_api_key: str = os.getenv("AZURE_OPENAI_API_KEY"),
    azure_api_version: str = os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature: float = 0
):
    """
    Initialize Azure OpenAI client
    
    Args:
        azure_deployment_name (str): Azure deployment name
        azure_endpoint (str): Azure endpoint URL
        azure_api_key (str): Azure API key
        azure_api_version (str): Azure API version
        temperature (float): Temperature for generation
        
    Returns:
        AzureChatOpenAI: Initialized Azure OpenAI client
    """
    return AzureChatOpenAI(
        azure_deployment=azure_deployment_name,
        azure_endpoint=azure_endpoint,
        api_key=azure_api_key,
        api_version=azure_api_version,
        temperature=temperature
    )


llm = initialize_azure_openai()

print("LLM initialized successfully.")

model = "gpt-4o"

In [ ]:
from langchain_anthropic import ChatAnthropic

def initialize_claude(
    api_key: str = os.getenv("ANTHROPIC_API_KEY"),
    model_name: str = os.getenv("ANTHROPIC_MODEL_NAME"),
    temperature: float = 0,
    max_tokens: int = 8192,  
    streaming: bool = True
):
    """
    Initialize Claude client using LangChain
    
    Args:
        api_key (str): Anthropic API key
        model_name (str): Model name to use (claude-3-7-sonnet-20250219 for latest Claude 3.7 Sonnet)
        temperature (float): Temperature for generation
        
    Returns:
        ChatAnthropic: Initialized Claude client
    """
    return ChatAnthropic(
        model=model_name,
        anthropic_api_key=api_key,
        temperature=temperature,
        max_tokens=max_tokens,
        streaming=streaming
    )

llm = initialize_claude()

print("Claude LLM initialized successfully.")

model = "claude-3-7"

In [ ]:
# Define state structure
class GraphState(dict):
    """State for the retrieval graph."""
    summary_text: Optional[str] = None  # embedding for vector search of past case retrieval
    case_json_full: Dict[str, Any]  # full case json                
    final_assessment: Optional[str] = None  

## Step 2: Utility Functions

These utility functions handle data processing tasks throughout the pipeline:

1. **Case Loading**:
   - Functions to load and preprocess case JSON files
   

In [ ]:
# Initial loading of JSON file (new case)
def load_case_from_json(file_path: str) -> Dict[str, Any]:
    """
    Load a FULL case from a JSON file as NEW case. 

    Args:
        file_path (str): Path to the JSON file

    Returns:
        Dict[str, Any]: FULL Case data, with "Case Disposition" removed(required for test/val set).
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        case_data = json.load(f)

    # Remove sections if it exists, required for ground_truth set 
    if "Case Disposition" in case_data:
        case_data.pop("Case Disposition")
    if "Ground Truth" in case_data:
        case_data.pop("Ground Truth")

    return case_data

## RAG Pipeline Implementation

The following sections implement the core RAG pipeline using a modular node-based approach. Each node performs a specific function and passes its results to subsequent nodes through the shared state object.

## Step 3: Node Definitions


### Core Pipeline Components

The pipeline consists of three main stages:
1. **Preprocessing**: Analyzing and summarizing the input case
2. **Generation**: Producing the final triage assessment

### 3.1: Preprocessing Node
The preprocessing node converts the raw case data into a concise clinical summary that captures the most relevant medical information. This summary serves as the query for both guideline retrieval and case similarity search, ensuring that retrieval focuses on clinically significant elements.

In [ ]:
# Define node to preprocess case data
def process_case_summary(state: GraphState) -> GraphState:
    
    ## Create Summary for VectorDB retrieval 
    
    sections_to_expand = {}
    
    sections_for_summary = {}
    if 'Demographics' in state["case_json_full"]:
        if 'Age' in state["case_json_full"]['Demographics']:
            sections_for_summary['Age'] = state["case_json_full"].get("Demographics", {}).get("Age", "")
        if 'Sex' in state["case_json_full"]['Demographics']:
            sections_for_summary['Sex'] = state["case_json_full"].get("Demographics", {}).get("Sex", "")
    if 'Vitals and Observations' in state["case_json_full"]:
        sections_for_summary['Vitals and Observations'] = state["case_json_full"].get("Vitals and Observations", {})
    if "Clinical Presentation" in state["case_json_full"]:
        sections_for_summary['Clinical Presentation'] = state["case_json_full"].get("Clinical Presentation", {})
        
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", """
        You are a specialized medical summarization assistant for emergency triage notes. Your task is to analyze structured JSON data from triage notes and create a concise, retrievable summary highlighting only the most important clinical information.
        ## INPUT STRUCTURE
        You will receive structured JSON data containing:
        - Demographics (sex, age)
        - Clinical Presentation (chief complaint, condition on arrival)
        - Vitals and Observations

        ## OUTPUT REQUIREMENTS
        1. Generate a concise (2-3 sentence) summary focusing on:
           - Patient demographics (sex, age)
           - Key presenting complaints
           - Abnormal physical findings
           - ONLY abnormal vital signs according to strict criteria

        2. Write in a factual, medical style optimized for vector embedding retrieval.

        ## MEDICAL TERMINOLOGY GUIDELINES
        - Interpret medical abbreviations contextually 
        - Fix obvious typos/misspellings based on medical context
        - Maintain medical precision while expanding clarity
        - For Hong Kong hospital/facility abbreviations:
          - "PMH" → "Princess Margaret Hospital"
          - "KCH" → "Kwai Chung Hospital"
          - "OAH" → "Old Age Home"
          - If uncertain about any abbreviation, do NOT expand it

        ## VITAL SIGNS ABNORMALITY CRITERIA
        ONLY mention vital signs that meet these specific abnormality thresholds:

        - Blood Pressure (Adult):
          - Hypertension: SBP > 180 mmHg or DBP > 100 mmHg
          - Hypotension: SBP < 90 mmHg

        - Heart Rate (Adults):
          - Tachycardia: HR > 120/min
          - Bradycardia: HR < 50/min

        - Temperature (Adult):
          - Fever: Temp ≥ 38°C
          - Hypothermia: Temp < 35°C

        - Blood Sugar:
          - Hyperglycemia: H'stix > 16 mmol/l
          - Hypoglycemia: H'stix < 3.9 mmol/l

        - Respiratory Status (always consider):
          - Oxygen supplementation (if present)
          - SpO2 level if abnormal
          - Abnormal respiratory rate

        - Conscious Level:
          - GCS score if not 15/15
          - Pupil abnormalities (if provided)

        - Pregnancy/Menstruation: Note if pregnant or abnormal menstrual patterns

        ## EXAMPLES OF GOOD SUMMARIES:

        ### Example 1:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "F",
                    "Age": "56 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "R/F ? IHD, tachycardia oedema\\nSOB , LL edema and neck swelling x 10/7 \\nchest discomfort - , palpitiation",
                    "Condition on Arrival": "speak in sentence"
                }},
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "E: 4, V: 5, M: 6, Score: 15/15",
                        "BP": "1st: 144/89 mmHg",
                        "PR/AR": "99",
                        "Temp": "36.2",
                        "SpO2": "100% on room air, --",
                        "RR": "--",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }},
                    "Menstruation / Pregnancy Status": "unspecified"
                }}
            }}
        ]
        
        Output: "56-year-old female presenting with shortness of breath, lower limb edema, and neck swelling for 10 days, accompanied by chest discomfort and palpitations. Patient can speak in full sentences and has been referred for evaluation of possible ischemic heart disease with tachycardia and edema. All vital signs are within normal ranges."

        ### Example 2:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "M",
                    "Age": "64 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "SOB this morning 0730\\nroom air 70%",
                    "Condition on Arrival": "no lower limb oedema\\nsputum sound+/-"
                }},    
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "",
                        "BP": "1st: 165/97 mmHg",
                        "PR/AR": "130",
                        "Temp": "38.1",
                        "SpO2": "96% on 6 L/min via Nasal Cannula",
                        "RR": "20",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }}
                }}
            }}
        ]
        Output: "64-year-old male presenting with sudden onset shortness of breath at 7:30 AM with significant hypoxia (70% on room air). Patient has tachycardia (HR 130), fever (38.1°C), and requires oxygen supplementation (6 L/min via Nasal Cannula). Clinical examination reveals sputum sounds without lower limb edema."

        ## IMPORTANT REMINDERS:
        - Focus on clinical relevance and retrieval effectiveness
        - Maintain medical accuracy while being concise
        - Only highlight abnormal vitals based on specific thresholds
        - Fix obvious errors but maintain meaning integrity
        - Adjust abnormality thresholds for pediatric cases
        """),
        ("user", """
        Case Data (JSON):
        {case_data}

        OUTPUT (Summary):
        """)
    ])
        
    # Prepare the input for the prompt
    case_data_str = json.dumps(sections_for_summary, indent=2)
    input_values = {
        "case_data": case_data_str
    }
    # Get the LLM response
    chain = summary_prompt | llm | StrOutputParser()
    response = chain.invoke(input_values)
    
    print("============ update summary_text ============")
    print(response)
    updated_state = state.copy()
    updated_state["summary_text"] = response
    return updated_state

### 3.2 Prediction node

1. TRIAGE DECISION based on TRIAGE CATEGORY DEFINITIONS


In [ ]:
def predict_triage_category(state: GraphState) -> GraphState:
    """
    Predict the triage category based on the case description and retrieved guideline sections
    using a stepwise approach and state-of-the-art RAG techniques.
    
    Args:
        state: Current state with case description and retrieved guideline content
        
    Returns:
        Updated state with the final triage category assessment
    """
    # Create a prompt that uses the retrieved guideline sections as context
    # Using CoT + RECITE RAG prompting techniques that combine Chain-of-Thought reasoning with
    # explicit citation of sources and direct quotes from the reference material
    # Based on research from "Lost in the Middle" (2023) and "RECITE: Recitation-Augmented Retrieval-Augmented Generation" (2024)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an emergency medicine triage specialist. Your task is to determine the appropriate triage category for a patient case based on the official triage category definitions.

        TRIAGE CATEGORY DEFINITIONS:
        | Triage Category | Patient Conditions | Actions of Staff | Target Response Time (Time from registration to medical consultation) |
        |---------------|------------------|----------------|---------------------------------------------------------------------|
        | 1 Critical | • Suffer from a life-threatening condition(s) caused by a major event • With unstable vital signs requiring immediate resuscitation | • Direct patient to resuscitation room • Attend patient immediately by a team comprising medical and nursing staff | • Immediate • 100% of cases within the target response time |
        | 2 Emergency | • Suffer from a potentially life-threatening condition • Borderline vital signs but with potential risk of rapid deterioration • Require emergency treatment and immediate continuous close monitoring | • Direct patient to resuscitation room / treatment cubicle • Offer medical attention and immediate continuous close monitoring within 15 mins. | • < 15 mins • 95% of cases within the target response time |
        | 3 Urgent | • Suffer from a major condition with potential risk of deterioration • Stable vital signs | • Direct patient to cubicle | • < 30 mins • 90% of cases within the target response time |
        | 4 Semi-urgent | • Suffer from acute but stable condition(s) • Stable vital signs • Can afford to wait some time without serious complications | • Direct patient to cubicle / walk-in clinic | • < 120 mins |
        | 5 Non-urgent | • Suffer from minor and stable condition(s) (including acute and non-acute conditions) • Can afford to wait without deterioration • Stable vital signs | • Direct patient to walk-in clinic | Remarks: • Conditions can be treated in primary health care facilities • Should be based on clinical judgement only. Economic, social factors and availability of facilities should not be taken into consideration. |

        Based on the patient case provided, determine the most appropriate triage category by matching the patient's condition to the definitions above.

        Provide your response in this format:

        **CLINICAL ASSESSMENT:**

        **FINAL TRIAGE CATEGORY:** [1/2/3/4/5]

        **RATIONALE:**
        [Explain which triage definition criteria the patient meets and why this category is most appropriate]

        """),
        ("user", """
        Current Case:
        {case_summary}
        
        Please determine the appropriate triage category based on the triage definitions provided.
        """)
    ])
    
    case_summary = f"""
    Clinical Summary: {state['summary_text']}
    
    Demographics: {state['case_json_full'].get('Demographics', {})}
    """
        
    # Prepare the input for the prompt
    input_values = {
        "case_summary": case_summary
    }

    # Get the LLM response with temperature 0 for consistency in assessments
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke(input_values)
    
    print("Triage Category Prediction:")
    print(response)
    print('='*80)
    
    updated_state = state.copy()

    # Update the state with the final assessment
    updated_state["final_assessment"] = response
    
    return updated_state

### Pipeline Assembly

The following functions assemble the individual nodes into a complete processing pipeline using LangGraph. The graph structure defines:
1. The execution order of processing steps
2. Data flow between components
3. State management throughout the pipeline

In [ ]:
def build_triage_graph():
    """
    Build a complete triage assessment graph using LangGraph.
    
    Args:
        case_json_dir (str): Directory containing the case JSON files
    
    Returns:
        The constructed graph
    """
    # Initialize the graph
    graph = StateGraph(GraphState)
    
    
    # Add nodes
    graph.add_node("process_case_summary", process_case_summary)
    graph.add_node("predict_triage_category", predict_triage_category)
    
    
    # Define edges
    graph.add_edge("process_case_summary", "predict_triage_category")
    graph.add_edge("predict_triage_category", END)
    
    # Set the entry point
    graph.set_entry_point("process_case_summary")
    
    # Compile the graph
    return graph.compile()

In [ ]:
triage_graph = build_triage_graph()

In [ ]:
triage_graph

## Step 4: Evaluation Framework


In [ ]:
def process_case_file(file_path: str) -> Dict[str, Any]:
    """
    Process a case file for complete triage assessment.
    
    Args:|
        file_path (str): Path to the case file (json)
        
    Returns:
        Dict[str, Any]: Complete state from the triage graph execution
    """
    # Load and format the case
    case_data = load_case_from_json(file_path)
    
    # Print the formatted case for inspection
    print("Case Content:")
    print(case_data)
    print("-" * 80)
    
    # Run the triage system
    initial_state = GraphState(
        case_json_full=case_data
    )

    # Return the complete result state, not just specific fields
    return triage_graph.invoke(initial_state)

## Single Case Testing

This function allows testing the system on individual cases to examine the detailed processing steps and outputs.

In [ ]:
process_case_file(file_path="test_case_file_path")

## Batch Evaluation

This section implements a framework for processing multiple cases and saving the results for systematic evaluation. The outputs include:
1. Case summaries
2. Retrieved guideline sections
3. Similar past cases
4. Complete triage assessments with reasoning

These structured outputs can be analyzed to evaluate the system's performance across different case types and triage categories.

In [ ]:
def batch_process_directory(
    input_directory: str,
    output_directory: str,
    max_files: int = None,
    verbose: bool = True
) -> int:
    """
    Process all JSON files in a directory and its subdirectories and save key elements to individual JSON files.
    Skips files that already have corresponding output files.
    
    Args:
        input_directory (str): Path to directory containing subdirectories with JSON case files
        output_directory (str): Directory where output files will be saved
        max_files (int, optional): Maximum number of files to process, useful for testing
        verbose (bool, optional): Whether to print detailed progress
        
    Returns:
        int: Number of files successfully processed
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_directory, exist_ok=True)
    
    # Find all JSON files recursively in the directory structure
    json_files = []
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.endswith('.json'):
                json_files.append(os.path.join(root, file))
    
    if max_files:
        json_files = json_files[:max_files]
    
    print(f"Found {len(json_files)} JSON files to process")
    
    # Track statistics
    processed_count = 0
    skipped_count = 0
    error_count = 0
    
    # Process each file with progress bar
    for file_path in tqdm(json_files, desc="Processing cases"):
        try:
            # Get relative path to preserve folder structure
            rel_path = os.path.relpath(file_path, input_directory)
            file_name = os.path.basename(file_path)
            
            # Create base filename without extension for output
            base_name = os.path.splitext(file_name)[0]
            
            # Create subdirectory in output directory to match input structure
            rel_dir = os.path.dirname(rel_path)
            output_subdir = os.path.join(output_directory, rel_dir)
            os.makedirs(output_subdir, exist_ok=True)
            
            # Check if output file already exists
            output_file_path = os.path.join(output_subdir, f"{base_name}_output.json")
            if os.path.exists(output_file_path):
                if verbose:
                    print(f"\nSkipping {rel_path} - output file already exists")
                skipped_count += 1
                continue
            
            if verbose:
                print(f"\nProcessing: {rel_path}")
            
            # Process the case
            state = process_case_file(file_path)
            
            # Save specific state components to individual files
            output_data = {
                "summary_text": state.get("summary_text"),
                "case_json_full": state.get("case_json_full"),
                "selected_guideline_sections": state.get("selected_guideline_sections"),
                "retrieved_cases_content": state.get("retrieved_cases_content"),
                "final_assessment": state.get("final_assessment")
            }
            
            # Save the combined output data to a single JSON file
            with open(output_file_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)
            
            processed_count += 1
            
            if verbose:
                print(f"Saved output to: {output_file_path}")
                print(f"Triage Category: {state.get('triage_category', None)}")
        
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            print(traceback.format_exc())
            error_count += 1
    
    # Print summary statistics
    print("\nProcessing Complete")
    print(f"Total files found: {len(json_files)}")
    print(f"Files processed: {processed_count}")
    print(f"Files skipped (already exist): {skipped_count}")
    print(f"Files with errors: {error_count}")
    
    return processed_count

In [ ]:
batch_process_directory(
    input_directory = "test_set_directory",
    output_directory = f"output/prediction_results/{model}/{model}_basic_prompt/1stTrial",
    max_files = None,
    verbose = True
)